# 06 Makemore WaveNet, RNN, And LSTM Comparison

This notebook turns the WaveNet hierarchy blog into a runnable experiment.

The question is deliberately narrow:

```text
same names dataset
same fixed 8-character context
same next-character prediction target
compare several sequence models by train/dev loss
```

The models:

```text
flat MLP                         baseline from earlier makemore work
hierarchical MLP                 visible pair-summary hierarchy
residual/skip WaveNet-style CNN  causal convolutions + gated residual blocks
scratch RNN                      recurrent hidden state baseline
scratch LSTM                     recurrent baseline with gates and cell state
```

The goal is not to chase the best possible name generator. The goal is to make the architectural differences measurable.


## Setup

This follows the local Makemore notebook style: load `names.txt`, build fixed context windows, then train with compact mini-batches.

The reference `makemore.py` pattern is useful here because every model exposes the same interface:

```python
logits, loss = model(idx, targets)
```

That keeps the experiment loop identical across MLP, WaveNet, RNN, and LSTM.


In [ ]:
from pathlib import Path
import math
import os
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd()
if not (ROOT / "data" / "names.txt").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "names.txt"
assert DATA_PATH.exists(), f"Could not find {DATA_PATH}"

# Keep this notebook reproducible and CPU-friendly.
# MPS/CUDA can be used manually, but CPU makes the comparison easier to reproduce.
device = torch.device("cpu")
torch.manual_seed(2147483647)
random.seed(42)

print("torch", torch.__version__)
print("device", device)


## Dataset

Each training example is still a fixed context window.

For `block_size = 8`:

```text
context characters -> next character
........           -> e
.......e           -> m
......em           -> m
.....emm           -> a
```

The model always receives 8 token ids and predicts one next token id.


In [ ]:
words = DATA_PATH.read_text().splitlines()
chars = sorted(set("".join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
block_size = 8

def build_dataset(words, block_size=block_size):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

shuffled = words[:]
random.Random(42).shuffle(shuffled)
n1 = int(0.8 * len(shuffled))
n2 = int(0.9 * len(shuffled))

Xtr, Ytr = build_dataset(shuffled[:n1])
Xdev, Ydev = build_dataset(shuffled[n1:n2])
Xte, Yte = build_dataset(shuffled[n2:])

print("words", len(words))
print("vocab_size", vocab_size)
print("train", tuple(Xtr.shape), tuple(Ytr.shape))
print("dev  ", tuple(Xdev.shape), tuple(Ydev.shape))
print("test ", tuple(Xte.shape), tuple(Yte.shape))


In [ ]:
for x, y in zip(Xtr[:8], Ytr[:8]):
    context = "".join(itos[int(i)] for i in x)
    target = itos[int(y)]
    print(f"{context} -> {target}")


## Shared Experiment Utilities

All models use the same training loop and evaluation function.

The comparison is not perfectly parameter-matched, so we print parameter counts next to the losses. The fair reading is:

```text
architecture + parameter count + optimizer budget -> observed loss
```

not:

```text
architecture alone caused everything
```


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def estimate_loss(model, X, Y, *, batch_size=4096, max_examples=None):
    was_training = model.training
    model.eval()

    if max_examples is not None:
        X = X[:max_examples]
        Y = Y[:max_examples]

    total_loss = 0.0
    total_count = 0
    for start in range(0, X.shape[0], batch_size):
        end = min(start + batch_size, X.shape[0])
        xb = X[start:end].to(device)
        yb = Y[start:end].to(device)
        _, loss = model(xb, yb)
        count = end - start
        total_loss += loss.item() * count
        total_count += count

    if was_training:
        model.train()
    return total_loss / total_count

def train_model(
    name,
    model,
    Xtr,
    Ytr,
    Xdev,
    Ydev,
    *,
    steps,
    batch_size,
    lr,
    eval_every,
    seed,
    grad_clip=1.0,
):
    model = model.to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    g = torch.Generator(device="cpu").manual_seed(seed)
    history = []
    t0 = time.time()

    for step in range(1, steps + 1):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
        xb = Xtr[ix].to(device)
        yb = Ytr[ix].to(device)

        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        if step == 1 or step % eval_every == 0 or step == steps:
            train_loss = estimate_loss(model, Xtr, Ytr, max_examples=20_000)
            dev_loss = estimate_loss(model, Xdev, Ydev)
            history.append({
                "step": step,
                "train_loss": train_loss,
                "dev_loss": dev_loss,
                "batch_loss": loss.item(),
            })
            print(
                f"{name:24s} step {step:5d}/{steps} "
                f"batch {loss.item():.4f} train {train_loss:.4f} dev {dev_loss:.4f}"
            )

    elapsed = time.time() - t0
    return history, elapsed

def print_results_table(rows):
    header = f"{'model':28s} {'params':>10s} {'train':>8s} {'dev':>8s} {'test':>8s} {'sec':>8s}"
    print(header)
    print("-" * len(header))
    for row in sorted(rows, key=lambda r: r["dev_loss"]):
        print(
            f"{row['name']:28s} {row['params']:10d} "
            f"{row['train_loss']:8.4f} {row['dev_loss']:8.4f} "
            f"{row['test_loss']:8.4f} {row['elapsed_sec']:8.1f}"
        )


## Small Makemore Layers

The hierarchy blog used hand-written layers to make the shapes visible. We keep that idea here, but wrap them as `nn.Module`s so they work with PyTorch optimizers.


In [ ]:
class FlattenConsecutive(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.n = n

    def forward(self, x):
        B, T, C = x.shape
        x = x.view(B, T // self.n, C * self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)
        return x

class MakemoreBatchNorm1d(nn.Module):
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        self.register_buffer("running_mean", torch.zeros(dim))
        self.register_buffer("running_var", torch.ones(dim))

    def forward(self, x):
        if self.training:
            if x.ndim == 2:
                reduce_dim = 0
            elif x.ndim == 3:
                reduce_dim = (0, 1)
            else:
                raise ValueError(f"expected 2D or 3D input, got shape {tuple(x.shape)}")

            xmean = x.mean(reduce_dim, keepdim=True)
            xvar = x.var(reduce_dim, keepdim=True, unbiased=False)

            with torch.no_grad():
                self.running_mean.mul_(1 - self.momentum).add_(self.momentum * xmean.squeeze())
                self.running_var.mul_(1 - self.momentum).add_(self.momentum * xvar.squeeze())
        else:
            view_shape = (1,) * (x.ndim - 1) + (-1,)
            xmean = self.running_mean.view(view_shape)
            xvar = self.running_var.view(view_shape)

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma.view((1,) * (x.ndim - 1) + (-1,)) * xhat + self.beta.view((1,) * (x.ndim - 1) + (-1,))

def loss_from_logits(logits, targets):
    loss = None if targets is None else F.cross_entropy(logits, targets)
    return logits, loss


## Baselines: Flat MLP And Hierarchical MLP

The flat MLP immediately collapses all 8 positions:

```text
[B, 8, n_embd] -> [B, 8 * n_embd]
```

The hierarchical MLP keeps the visible pair-summary pattern from the blog:

```text
8 positions -> 4 summaries -> 2 summaries -> 1 summary
```


In [ ]:
class FlatMLP(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=24, n_hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, n_embd),
            nn.Flatten(),
            nn.Linear(block_size * n_embd, n_hidden, bias=False),
            MakemoreBatchNorm1d(n_hidden),
            nn.Tanh(),
            nn.Linear(n_hidden, vocab_size),
        )
        with torch.no_grad():
            self.net[-1].weight.mul_(0.1)
            self.net[-1].bias.zero_()

    def forward(self, idx, targets=None):
        logits = self.net(idx)
        return loss_from_logits(logits, targets)

class HierarchicalMLP(nn.Module):
    def __init__(self, vocab_size, n_embd=24, n_hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, n_embd),
            FlattenConsecutive(2), nn.Linear(n_embd * 2, n_hidden, bias=False), MakemoreBatchNorm1d(n_hidden), nn.Tanh(),
            FlattenConsecutive(2), nn.Linear(n_hidden * 2, n_hidden, bias=False), MakemoreBatchNorm1d(n_hidden), nn.Tanh(),
            FlattenConsecutive(2), nn.Linear(n_hidden * 2, n_hidden, bias=False), MakemoreBatchNorm1d(n_hidden), nn.Tanh(),
            nn.Linear(n_hidden, vocab_size),
        )
        with torch.no_grad():
            self.net[-1].weight.mul_(0.1)
            self.net[-1].bias.zero_()

    def forward(self, idx, targets=None):
        logits = self.net(idx)
        return loss_from_logits(logits, targets)

def trace_sequential_shapes(model, idx):
    x = idx
    for layer in model.net:
        x = layer(x)
        print(f"{layer.__class__.__name__:22s} {tuple(x.shape)}")
    return x

probe = Xtr[:4]
print("Flat MLP")
trace_sequential_shapes(FlatMLP(vocab_size, block_size), probe)
print("\nHierarchical MLP")
trace_sequential_shapes(HierarchicalMLP(vocab_size), probe)


## Residual/Skip WaveNet-Style Model

This is the architectural step beyond the visible hierarchy.

The WaveNet-style model keeps a feature vector at every time position:

```text
[B, T] -> embedding -> [B, T, n_embd]
```

Then convolution code prefers channel-first layout:

```text
[B, T, n_embd] -> [B, n_embd, T]
```

Each residual block does this:

```text
x -> causal dilated convs -> gate -> z
z -> residual projection -> update -> x_next = x + update
z -> skip projection     -> skip contribution for final logits
```

The skip outputs from all depths are summed before the output head.


In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
        )

    def forward(self, x):
        x = F.pad(x, (self.left_padding, 0))
        return self.conv(x)

class ResidualBlock(nn.Module):
    def __init__(self, residual_channels, skip_channels, dilation):
        super().__init__()
        self.dilation = dilation
        self.filter_conv = CausalConv1d(residual_channels, residual_channels, kernel_size=2, dilation=dilation)
        self.gate_conv = CausalConv1d(residual_channels, residual_channels, kernel_size=2, dilation=dilation)
        self.residual_projection = nn.Conv1d(residual_channels, residual_channels, kernel_size=1)
        self.skip_projection = nn.Conv1d(residual_channels, skip_channels, kernel_size=1)

    def forward(self, x):
        candidate = torch.tanh(self.filter_conv(x))
        gate = torch.sigmoid(self.gate_conv(x))
        z = candidate * gate

        update = self.residual_projection(z)
        skip = self.skip_projection(z)
        x_next = x + update
        return x_next, skip

class ResidualSkipWaveNet(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd=24,
        residual_channels=128,
        skip_channels=128,
        dilations=(1, 2, 4, 1, 2, 4),
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.input_projection = nn.Conv1d(n_embd, residual_channels, kernel_size=1)
        self.blocks = nn.ModuleList([
            ResidualBlock(residual_channels, skip_channels, dilation=d)
            for d in dilations
        ])
        self.output_head = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(skip_channels, skip_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(skip_channels, vocab_size, kernel_size=1),
        )
        with torch.no_grad():
            self.output_head[-1].weight.mul_(0.1)
            self.output_head[-1].bias.zero_()

    def forward(self, idx, targets=None, return_debug=False):
        debug = []
        x = self.embedding(idx)                 # [B, T, n_embd]
        debug.append(("embedding", tuple(x.shape)))

        x = x.transpose(1, 2)                   # [B, n_embd, T]
        debug.append(("transpose", tuple(x.shape)))

        x = self.input_projection(x)            # [B, residual_channels, T]
        debug.append(("input_projection", tuple(x.shape)))

        skips = []
        for block_index, block in enumerate(self.blocks):
            x, skip = block(x)
            skips.append(skip)
            debug.append((f"block_{block_index}_dilation_{block.dilation}_residual", tuple(x.shape)))
            debug.append((f"block_{block_index}_dilation_{block.dilation}_skip", tuple(skip.shape)))

        combined_skip = torch.stack(skips).sum(dim=0)
        debug.append(("sum_skips", tuple(combined_skip.shape)))

        logits_all_positions = self.output_head(combined_skip).transpose(1, 2)  # [B, T, vocab]
        debug.append(("logits_all_positions", tuple(logits_all_positions.shape)))

        logits = logits_all_positions[:, -1, :]  # predict the token after the final context position
        debug.append(("last_position_logits", tuple(logits.shape)))

        logits, loss = loss_from_logits(logits, targets)
        if return_debug:
            return logits, loss, debug
        return logits, loss

wavenet_probe = ResidualSkipWaveNet(vocab_size)
_, _, debug = wavenet_probe(Xtr[:4], Ytr[:4], return_debug=True)
for name, shape in debug:
    print(f"{name:36s} {shape}")


## Scratch RNN And LSTM Models

These models deliberately do not use `nn.RNN` or `nn.LSTM`. They use ordinary PyTorch tensor operations and `nn.Linear` layers so the recurrent math is visible.

A scratch RNN reads the sequence one token at a time and carries a hidden state forward:

```text
h_t = RNN(x_t, h_{t-1})
```

The final hidden state becomes the summary of the whole 8-character context.

A scratch LSTM is a gated RNN. It carries both:

```text
h_t: visible hidden state
c_t: longer-term cell state
```

The LSTM gates decide what to forget, what to write, and what to expose. That makes it easier to preserve information across more steps than a plain RNN.


In [ ]:
class RNNCellScratch(nn.Module):
    def __init__(self, n_embd, n_hidden):
        super().__init__()
        self.xh_to_h = nn.Linear(n_embd + n_hidden, n_hidden)

    def forward(self, xt, hprev):
        xh = torch.cat([xt, hprev], dim=1)
        h = torch.tanh(self.xh_to_h(xh))
        return h

class RNNCharScratch(nn.Module):
    def __init__(self, vocab_size, n_embd=24, n_hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.cell = RNNCellScratch(n_embd, n_hidden)
        self.h0 = nn.Parameter(torch.zeros(1, n_hidden))
        self.output = nn.Linear(n_hidden, vocab_size)
        with torch.no_grad():
            self.output.weight.mul_(0.1)
            self.output.bias.zero_()

    def forward(self, idx, targets=None):
        emb = self.embedding(idx)          # [B, T, n_embd]
        B, T, _ = emb.shape
        h = self.h0.expand(B, -1)          # [B, n_hidden]

        for t in range(T):
            xt = emb[:, t, :]              # [B, n_embd]
            h = self.cell(xt, h)           # [B, n_hidden]

        logits = self.output(h)
        return loss_from_logits(logits, targets)

class LSTMCellScratch(nn.Module):
    def __init__(self, n_embd, n_hidden):
        super().__init__()
        self.xh_to_gates = nn.Linear(n_embd + n_hidden, 4 * n_hidden)

    def forward(self, xt, hprev, cprev):
        xh = torch.cat([xt, hprev], dim=1)
        gates = self.xh_to_gates(xh)
        forget, input_gate, candidate, output_gate = gates.chunk(4, dim=1)

        forget = torch.sigmoid(forget)
        input_gate = torch.sigmoid(input_gate)
        candidate = torch.tanh(candidate)
        output_gate = torch.sigmoid(output_gate)

        c = forget * cprev + input_gate * candidate
        h = output_gate * torch.tanh(c)
        return h, c

class LSTMCharScratch(nn.Module):
    def __init__(self, vocab_size, n_embd=24, n_hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.cell = LSTMCellScratch(n_embd, n_hidden)
        self.h0 = nn.Parameter(torch.zeros(1, n_hidden))
        self.c0 = nn.Parameter(torch.zeros(1, n_hidden))
        self.output = nn.Linear(n_hidden, vocab_size)
        with torch.no_grad():
            self.output.weight.mul_(0.1)
            self.output.bias.zero_()

    def forward(self, idx, targets=None):
        emb = self.embedding(idx)          # [B, T, n_embd]
        B, T, _ = emb.shape
        h = self.h0.expand(B, -1)
        c = self.c0.expand(B, -1)

        for t in range(T):
            xt = emb[:, t, :]
            h, c = self.cell(xt, h, c)

        logits = self.output(h)
        return loss_from_logits(logits, targets)

for model in [RNNCharScratch(vocab_size), LSTMCharScratch(vocab_size)]:
    logits, loss = model(Xtr[:4], Ytr[:4])
    print(f"{model.__class__.__name__:14s} logits {tuple(logits.shape)} loss {loss.item():.4f}")


## Training Budget

These defaults are intentionally modest. Increase `MAX_STEPS` to make the comparison stronger.

For command-line runs, you can override the budget without editing the notebook:

```bash
MAKEMORE_MAX_STEPS=200 MAKEMORE_EVAL_EVERY=50 jupyter nbconvert --execute ...
```

Good quick run:

```text
MAX_STEPS = 800
```

Better comparison:

```text
MAX_STEPS = 3000 or 5000
```

Because all models use the same data, batch size, optimizer, and evaluation function, the resulting table is directly useful even when the absolute losses are not final.


In [ ]:
N_EMBD = 24
N_HIDDEN = 128
BATCH_SIZE = 64
MAX_STEPS = int(os.environ.get("MAKEMORE_MAX_STEPS", 800))
EVAL_EVERY = int(os.environ.get("MAKEMORE_EVAL_EVERY", 200))
LEARNING_RATE = 3e-3

model_factories = {
    "flat_mlp": lambda: FlatMLP(vocab_size, block_size, n_embd=N_EMBD, n_hidden=N_HIDDEN),
    "hierarchical_mlp": lambda: HierarchicalMLP(vocab_size, n_embd=N_EMBD, n_hidden=N_HIDDEN),
    "residual_skip_wavenet": lambda: ResidualSkipWaveNet(
        vocab_size,
        n_embd=N_EMBD,
        residual_channels=N_HIDDEN,
        skip_channels=N_HIDDEN,
        dilations=(1, 2, 4, 1, 2, 4),
    ),
    "rnn_scratch": lambda: RNNCharScratch(vocab_size, n_embd=N_EMBD, n_hidden=N_HIDDEN),
    "lstm_scratch": lambda: LSTMCharScratch(vocab_size, n_embd=N_EMBD, n_hidden=N_HIDDEN),
}

for name, factory in model_factories.items():
    model = factory()
    print(f"{name:24s} params {count_parameters(model):,}")


## Run The Comparison

The first loss should be near `log(27) = 3.296`, because there are 27 possible next characters.

The useful signal is how far each model pushes dev loss down under the same training budget.


In [ ]:
results = []
histories = {}

for model_index, (name, factory) in enumerate(model_factories.items()):
    print("\n" + "=" * 80)
    print(name)
    model = factory()
    params = count_parameters(model)
    history, elapsed = train_model(
        name,
        model,
        Xtr,
        Ytr,
        Xdev,
        Ydev,
        steps=MAX_STEPS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
        eval_every=EVAL_EVERY,
        seed=1000 + model_index,
    )
    histories[name] = history

    train_loss = estimate_loss(model, Xtr, Ytr, max_examples=50_000)
    dev_loss = estimate_loss(model, Xdev, Ydev)
    test_loss = estimate_loss(model, Xte, Yte)
    results.append({
        "name": name,
        "params": params,
        "train_loss": train_loss,
        "dev_loss": dev_loss,
        "test_loss": test_loss,
        "elapsed_sec": elapsed,
        "model": model,
    })

print("\nFinal comparison")
print_results_table(results)


## Read The Result

Interpret the table in two passes.

First, compare `flat_mlp` and `hierarchical_mlp`:

```text
Does explicitly building local summaries help compared with flattening everything immediately?
```

Then compare `hierarchical_mlp` and `residual_skip_wavenet`:

```text
Does keeping positions alive, using causal convolution, and collecting skip evidence improve dev loss?
```

Then compare the convolutional model against `rnn_scratch` and `lstm_scratch`:

```text
Does recurrence or convolution work better under this small fixed-context budget?
```

For a rigorous claim, rerun with multiple seeds and a larger step budget. For learning the architecture, one run is enough to see the mechanics.


In [ ]:
for name, history in histories.items():
    compact = [(h["step"], round(h["dev_loss"], 4)) for h in history]
    print(f"{name:24s} {compact}")


## Sample From The Trained Models

Sampling is not the metric, but it is a good sanity check. The model should produce name-like strings instead of uniform random letters.


In [ ]:
@torch.no_grad()
def sample(model, num_names=10, max_len=20, seed=1234):
    model.eval()
    g = torch.Generator(device="cpu").manual_seed(seed)
    names = []
    for _ in range(num_names):
        context = [0] * block_size
        out = []
        for _ in range(max_len):
            xb = torch.tensor([context], dtype=torch.long, device=device)
            logits, _ = model(xb)
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs.cpu(), num_samples=1, generator=g).item()
            context = context[1:] + [ix]
            if ix == 0:
                break
            out.append(itos[ix])
        names.append("".join(out))
    return names

for row in sorted(results, key=lambda r: r["dev_loss"]):
    print("\n" + row["name"])
    print(" ".join(sample(row["model"], num_names=12, seed=2024)))


## Main Takeaways

The architectural distinction is now concrete:

```text
flat MLP:
flatten the whole context immediately

hierarchical MLP:
combine adjacent groups in a visible tree

WaveNet-style CNN:
slide shared local rules across positions, keep positions alive, use residual updates, sum skip evidence

scratch RNN:
read left to right and keep one recurrent hidden state

scratch LSTM:
read left to right with gates and a cell state for better memory
```

What to look for in your run:

```text
lower dev loss means better next-character prediction on unseen names
lower train but worse dev means overfitting
similar loss with fewer parameters means a more efficient architecture for this setup
```

The WaveNet residual/skip flow is the important structural idea. Residual routes keep deep computation trainable; skip routes let every depth contribute directly to the prediction.
